##Silver Transformation of the races bronze table
### 1. Read the table from bronze schema

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run ../00.Common/02.Helper_Notebook

In [0]:
source_name = f"{catalog_name}.{bronze_schema}.races"
target_name = f"{catalog_name}.{silver_schema}.races"

In [0]:
races_df = spark.table(source_name)
display(races_df)

### 2. Keep only the required columns for analysis 

In [0]:
from pyspark.sql import functions as F
races_required_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("SourceFile")
    )

### 3. Rename the columns

In [0]:
races_renamed_df = races_required_df.withColumnsRenamed({"raceName":"race_name","circuitId":"circuit_id","date":"race_date"})
display(races_renamed_df)

### 4. Removing duplicates and NULL from the dataset

In [0]:
# Removing the null values using sql and column expressions
# circuits_clean_df = circuits_renamed_df.filter(
#     "circuit_id IS NOT NULL"
# )
# races_clean_df = races_renamed_df.filter(F.col("circuits_id").isNotNull())

In [0]:
# Removing duplicates based on the primary key
races_clean_df = races_renamed_df.dropDuplicates(["season","round"])
display(races_clean_df)

### 5. Transforming the column values 

In [0]:
# Converting the values to initcap format in locality and circuit name columns
races_final_df = (races_clean_df
                     .withColumn("race_name",F.initcap(F.col("race_name")))
)
display(races_final_df)

### 6. Writing the final dataframe as table into the silver schema

In [0]:
(
    races_final_df.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(target_name)
)

In [0]:
%sql
select * from formula1.silver.races;